# REPASO ENTRENAMIENTO REFORZADO MONTE CARLO ON POLICY


In [149]:
import copy
import numpy as np

In [210]:
class Instancia():
    def __init__(self,largo = int ,ancho = int):
        self.largo = largo
        self.ancho = ancho

        self.partida = { 'x': self.ancho/2, 'y': 0 }
        self.meta = { 'x': self.ancho/2, 'y': self.largo }

    def __str__(self):
            return f'Largo: {self.largo}, Ancho: {self.ancho}, Partida: {self.partida}, Meta: {self.meta}'


class Estado():
    def __init__(self,x,y):
        self.x = x
        self.y = y
    
    def __str__(self):
        return f'x: {self.x}, y: {self.y}'

    def __eq__(self, otro_estado):
        return self.x == otro_estado.x and self.y == otro_estado.y

    def __hash__(self):
        return hash((self.x, self.y))


class Accion():
    def __init__(self, direccion_x = int, direccion_y = int):
        self.direccion_x = direccion_x
        self.direccion_y = direccion_y

    def __str__(self):
        return f'Direccion x: {self.direccion_x}, Direccion y: {self.direccion_y}'

    def __eq__(self, otra_accion):
        return self.direccion_x == otra_accion.direccion_x and self.direccion_y == otra_accion.direccion_y

    def __hash__(self):
        return hash((self.direccion_x, self.direccion_y))


class Proceso():
    def __init__(self,instancia):
        self.instancia = instancia

    def obtener_estado_inicial(self):
        '''
        Método que retorna el estado inicial del problema, leyendo la instancia
        args:
            None
        return:
                Estado
        '''
        return Estado(x = self.instancia.partida['x'], y = self.instancia.partida['y'])
    

    def transicion(self, estado = Estado, accion = Accion):
        '''
        Método que retorna el estado y recompensa siguiente al aplicar una acción a un estado
        args:
            estado: Estado
            accion: Accion
        return:
                Estado
        '''

        nuevo_estado = copy.deepcopy(estado)

        nuevo_estado.x += accion.direccion_x

        nuevo_estado.y += accion.direccion_y

        recompensa = self.obtener_recompensa(nuevo_estado)

        return nuevo_estado, recompensa



    def obtener_recompensa(self, estado = Estado):
        '''
        Método que retorna la recompensa de un estado
        args:
            estado: Estado
        return:
                int
        '''
        # si llega a la meta
        if estado.x == self.instancia.meta['x'] and estado.y == self.instancia.meta['y']:
            return 10000
        # si se sale de los margenes
        elif 0<= estado.x <= self.instancia.ancho and 0<= estado.y <= self.instancia.largo:
            return -0.1
        
        # si sigue en el mapa pero no llega aún
        else: 
            return -100





class Politica():
    def __init__(self,proceso, instancia):
        self.proceso = proceso
        self.instancia = instancia
        self.acciones =  None
        self.estados_posibles = None
        self.obtener_acciones()
        self.Q_sa = None  # {Estado: {Accion: Q(s,a)}}
        self.obtener_Q_sa_arbitrario()
        self.sa_n = None
        self.inicializar_sa_n()


    def inicializar_sa_n(self):
        '''
        Método que inicializa el diccionario sa_n
        args:
            None
        return:
            None
        '''
        self.sa_n = {estado: {accion: 0 for accion in self.acciones} for estado in self.estados_posibles}



    def obtener_acciones(self):
        '''
        Método que retorna las acciones posibles en un estado
        args:
            None
        return:
            None
        '''
        # movimientos posibles
        movimientos = [-1, 0 ,1]

        self.acciones= [ Accion( direccion_x= i,  direccion_y= j)  for i in movimientos for j in movimientos]


    def obtener_Q_sa_arbitrario(self):
        '''
        Método que retorna la función de valor Q(s,a) de forma arbitraria
        args:
            None
        return:
            None
        '''

        posiciones_posibles_en_x = [i for i in range(0,self.instancia.ancho+1)]
        posiciones_posibles_en_y = [i for i in range(0,self.instancia.largo+1)]

        estados_posibles = [Estado(x=i, y=j) for i in posiciones_posibles_en_x for j in posiciones_posibles_en_y]
        self.estados_posibles = estados_posibles

        self.Q_sa = {estado: {accion: 0 for accion in self.acciones} for estado in estados_posibles}


    def ejecutar_episodio_epsilon_greedy(self, epsilon = float):
        '''
        Método que ejecuta un episodio de aprendizaje por refuerzo usando la política epsilon-greedy
        args:
            epsilon: float
        return:
            trayectoria = Lista
        '''

        # obtenemos el estado inicial
        estado_actual = self.proceso.obtener_estado_inicial()

        trayectoria = []

        # mientras no lleguemos a la meta
        while True:
            # obtenemos una accion para el estado en el que estamos
            accion = self.epsilon_greedy(estado_actual, epsilon)
            nuevo_estado, recompensa = self.proceso.transicion(estado_actual, accion)
            trayectoria.append({'estado':estado_actual, 'accion' :accion, 'recompensa': recompensa})

            # vemos si el estado nuevo es terminal
            if recompensa == 1000 or recompensa == -100:
                break
            else:
                estado_actual = nuevo_estado
        return trayectoria

    def epsilon_greedy(self, estado, epsilon):
        '''
        Método que retorna una acción para un estado usando la política epsilon-greedy
        args:
            estado: Estado
            epsilon: float
        return:
            accion: Accion
        '''
        # obtenemmos la accion más conveniente según los Q valores
        valor_aleatorio = np.random.rand()
        if valor_aleatorio < epsilon:
            return np.random.choice(self.acciones)
        
        mejor_accion = None
        mejor_Q_valor = -np.inf
        for accion in self.acciones:
            if self.Q_sa[estado][accion] >= mejor_Q_valor:
                mejor_accion = accion
                mejor_Q_valor = self.Q_sa[estado][accion]
        
        return  mejor_accion

    def Monte_Carlo(self, epochs = int):
        '''
        Método que ejecuta un episodio de aprendizaje por refuerzo usando el método de Monte Carlo
        args:
            None
        return:
            None
        '''

        for _ in range(epochs):
            # obtenemos la trayectoria
            trayectoria = self.ejecutar_episodio_epsilon_greedy(epsilon = 0.2)
            G = 0
            for t  in reversed(trayectoria):
                G = t['recompensa'] + 0.9* G
                n = self.sa_n[t['estado']][t['accion']]
                self.Q_sa[t['estado']][t['accion']] = (n * self.Q_sa[t['estado']][t['accion']] + G) / (n + 1)
                self.sa_n[t['estado']][t['accion']] += 1


    def elegir_accion_optima(self, estado = Estado):
        acciones = self.acciones
        mayor_q_valor = -np.inf
        accion_elegida = None
        for accion in acciones:
            if self.Q_sa[estado][accion] >= mayor_q_valor:
                mayor_q_valor = self.Q_sa[estado][accion]
                accion_elegida = accion
        return accion_elegida
    

    def ejecutar_episodio_optimo(self):
        '''
        Método que ejecuta un episodio de aprendizaje por refuerzo usando la política óptima
        args:
            None
        return:
            trayectoria = Lista
        '''

        # obtenemos el estado inicial
        estado_actual = self.proceso.obtener_estado_inicial()
        trayectoria = []
        # mientras no lleguemos a la meta
        while True:
            # obtenemos una accion para el estado en el que estamos
            accion = self.elegir_accion_optima(estado_actual)
            nuevo_estado, recompensa = self.proceso.transicion(estado_actual, accion)
            trayectoria.append({'estado':estado_actual, 'accion': accion, 'recompensa': recompensa})
            # vemos si el estado nuevo es terminal
            if recompensa == 10000 or recompensa == -100:
                break
            else:
                estado_actual = nuevo_estado
        return trayectoria


In [211]:
instancia = Instancia( largo = 10, ancho = 10)

proceso = Proceso(instancia)

politica = Politica(proceso, instancia)

politica.Monte_Carlo(epochs = 3000)


In [214]:
trayectoria_optima = politica.ejecutar_episodio_optimo()

for t in trayectoria_optima:
    print(t['estado'])
    print(t['accion'])
    print('paso')

x: 5.0, y: 0
Direccion x: -1, Direccion y: 1
paso
x: 4.0, y: 1
Direccion x: 0, Direccion y: 1
paso
x: 4.0, y: 2
Direccion x: -1, Direccion y: 1
paso
x: 3.0, y: 3
Direccion x: 0, Direccion y: 1
paso
x: 3.0, y: 4
Direccion x: 1, Direccion y: 1
paso
x: 4.0, y: 5
Direccion x: -1, Direccion y: 1
paso
x: 3.0, y: 6
Direccion x: 1, Direccion y: 1
paso
x: 4.0, y: 7
Direccion x: 0, Direccion y: 1
paso
x: 4.0, y: 8
Direccion x: 1, Direccion y: 1
paso
x: 5.0, y: 9
Direccion x: 0, Direccion y: 1
paso
